# Data Cleaning 05 -- CFTC Commitment of Traders

## Input
`Data/Data_Collection/Initial/01_CTFC/01_cftc.parquet` (969 rows, weekly, 2006-06-13 to 2024-12-31)

## Purpose
Cleans weekly CFTC Commitments of Traders positioning data for S&P 500 E-mini futures. Keyed on `date` only (no PERMNO). Key concerns addressed: verifying the weekly structure and day-of-week consistency, checking the publication lag between report date and available date, validating net position arithmetic (net = long - short), and confirming the structural late start of the disaggregated report format.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification. Publication lag between `available_date` and `date` is computed and reported (expected: 6 days for every row -- positions as-of Tuesday, released the following Monday).

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with first/last valid dates
- Per-row NaN distribution and identification of worst rows

## Stage 2: Weekly Structure Checks
- **Day-of-week distribution:** confirms report dates are Tuesdays and available dates are Mondays
- **Date gap distribution:** reports gap statistics; 948 gaps of exactly 7 days, 10 gaps of 6 days and 10 of 8 days (holiday-shifted Tuesdays). No missing weeks.
- **Long gaps (>10 days):** identifies any unusual gaps
- **Duplicate date check**

## Stage 3: Value Range & Quality Checks
- **Summary statistics** for all numeric columns
- **Negative value check:** identifies which columns contain negative values (expected for net positions and spreads)
- **Ratio/percentage column identification**
- **Constant or near-constant column check**
- **Net position consistency:** verifies `lev_net = lev_long - lev_short`, `am_net = am_long - am_short`, `dealer_net = dealer_long - dealer_short` with zero error for all three trader types

## Stage 5: Clean & Save

### No Columns Dropped
All 22 factor columns retained. Component columns (long, short, spread) and net columns are both kept -- the model can select what is useful, and z-standardisation makes scale irrelevant.

### `available_date` Retained as Metadata
The publication lag is exactly 6 days for every row. The merge pipeline should use `available_date` (not `date`) when forward-filling to daily, to ensure point-in-time correctness: Tuesday's positions should only enter the model from the following Monday onwards.

### Data Starts June 2006, Not 2004
The CFTC Disaggregated report (which breaks out leveraged funds, asset managers, and dealers separately) only began in June 2006. Earlier CFTC data uses the legacy commercial/non-commercial breakdown which is not comparable. Same treatment as FRED daily `twexb`/`twexm` -- structural late start, resolved when the merged dataset starts from 2006.

### Structural NaN (2 Values Only)
`lev_net_chg` and `am_net_chg` are NaN on the first row (2006-06-13) -- no prior week to compute a change from. Left as NaN.

### No Winsorisation Applied
All value ranges are sensible for S&P 500 futures positioning. Leveraged funds are typically net short (928/969 weeks), asset managers typically net long (969/969 weeks). Winsorisation handled in the merge pipeline.

## Output
`Data/Data_Collection/Cleaned/05_CFTC/cftc_clean.parquet` -- 22 factor columns (all retained), plus `date` and `available_date`

In [5]:
# %% [markdown]
# # Data Cleaning: 01_cftc.parquet
#
# Source: Data/Data_Collection/Initial/01_CTFC/01_cftc.parquet
# Output: Data/Data_Collection/Cleaned/01_CFTC/cftc_clean.parquet
#
# Weekly CFTC Commitments of Traders data pulled via the Socrata API.
# No PERMNO — keyed on date only. Contains futures positioning data
# for S&P 500 E-mini futures (leveraged funds, asset managers, dealers).
#
# Key concerns:
#   - What day of the week? (CFTC reports are as-of Tuesday, released Friday)
#   - Are there gaps in the weekly series?
#   - available_date column tracks publication date (Friday release)
#   - Positioning data: long, short, spread, net positions by trader type

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/01_CTFC/01_cftc.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/05_CFTC')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — CFTC")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
df['available_date'] = pd.to_datetime(df['available_date'])

# Separate date columns from numeric factor columns
date_cols = ['date', 'available_date']
factor_cols = [c for c in df.columns if c not in date_cols]

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Available date range: {df['available_date'].min().date()} → {df['available_date'].max().date()}")
print(f"Unique dates: {df['date'].nunique():,}")

print(f"\nFactor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<30s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (5 rows) ---")
print(df.head(5).to_string(index=False))

print(f"\n--- Tail (5 rows) ---")
print(df.tail(5).to_string(index=False))

# ── Publication lag check ────────────────────────────────────────────────────
pub_lag = (df['available_date'] - df['date']).dt.days
print(f"\n--- Publication lag (available_date - date) ---")
print(f"  Mean: {pub_lag.mean():.1f} days")
print(f"  Median: {pub_lag.median():.0f} days")
print(f"  Min: {pub_lag.min():.0f} days, Max: {pub_lag.max():.0f} days")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN (sorted descending) ───────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<30s} {'NaN %':>8s}  {'Count':>6s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 75)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    valid = df[df[col].notna()]['date']
    first = valid.min().date() if len(valid) > 0 else 'N/A'
    last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<30s} {pct:>7.2f}%  {count:>6d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>6,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-5 NaN: {((row_nan >= 1) & (row_nan <= 5)).sum():>6,d}")
print(f"  Rows with >5 NaN: {(row_nan > 5).sum():>6,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

if (row_nan > 0).any():
    print(f"\n  5 rows with most NaN:")
    worst = df.loc[row_nan.nlargest(5).index, ['date']].copy()
    worst['n_nan'] = row_nan.nlargest(5).values
    worst['day_of_week'] = worst['date'].dt.day_name()
    print(worst.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: WEEKLY STRUCTURE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: WEEKLY STRUCTURE CHECKS")
print("=" * 90)

# ── 2a. Day-of-week distribution ────────────────────────────────────────────
print(f"\n--- Day-of-week distribution (report date) ---")
dow_counts = df['date'].dt.day_name().value_counts()
print(dow_counts.to_string())

print(f"\n--- Day-of-week distribution (available_date) ---")
dow_avail = df['available_date'].dt.day_name().value_counts()
print(dow_avail.to_string())

# ── 2b. Date gap analysis ───────────────────────────────────────────────────
print(f"\n--- Date gap distribution ---")
df_sorted = df.sort_values('date')
date_diffs = df_sorted['date'].diff().dt.days.dropna()
print(f"  Mean gap: {date_diffs.mean():.1f} days")
print(f"  Median gap: {date_diffs.median():.0f} days")
print(f"  Min gap: {date_diffs.min():.0f} days")
print(f"  Max gap: {date_diffs.max():.0f} days")
print(f"\n  Gap distribution:")
for gap, count in date_diffs.value_counts().sort_index().head(10).items():
    print(f"    {int(gap):>3d} days: {count:>5,d}")

# ── 2c. Long gaps (>10 days) ────────────────────────────────────────────────
long_gaps = date_diffs[date_diffs > 10]
if len(long_gaps) > 0:
    print(f"\n  Gaps > 10 days: {len(long_gaps)}")
    for idx in long_gaps.index[:10]:
        gap_end = df_sorted.loc[idx, 'date']
        prev_idx = df_sorted.index[df_sorted.index.get_loc(idx) - 1]
        gap_start = df_sorted.loc[prev_idx, 'date']
        print(f"    {gap_start.date()} → {gap_end.date()} ({int(date_diffs.loc[idx])} days)")
else:
    print(f"\n  ✓ No gaps > 10 days")

# ── 2d. Duplicate dates ─────────────────────────────────────────────────────
print(f"\n--- Duplicate dates ---")
n_dupes = df['date'].duplicated().sum()
if n_dupes == 0:
    print(f"  ✓ No duplicate dates")
else:
    print(f"  ⚠ {n_dupes} duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

# ── 3a. Summary stats for all numeric columns ───────────────────────────────
print(f"\n--- Summary statistics ---")
print(f"\n  {'Column':<30s} {'min':>14s}  {'median':>14s}  {'max':>14s}  {'mean':>14s}")
print("  " + "-" * 90)
for col in factor_cols:
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    if not pd.api.types.is_numeric_dtype(vals):
        print(f"  {col:<30s} (non-numeric, skipped)")
        continue
    print(f"  {col:<30s} {vals.min():>14,.2f}  {vals.median():>14,.2f}  "
          f"{vals.max():>14,.2f}  {vals.mean():>14,.2f}")

# ── 3b. Check for negative values ───────────────────────────────────────────
print(f"\n--- Columns with negative values ---")
for col in factor_cols:
    vals = df[col].dropna()
    if not pd.api.types.is_numeric_dtype(vals):
        continue
    n_neg = (vals < 0).sum()
    if n_neg > 0:
        print(f"  {col:<30s} {n_neg:>6d} negative values "
              f"(min: {vals.min():,.2f})")

# ── 3c. Check for ratio/percentage columns ──────────────────────────────────
print(f"\n--- Potential ratio/percentage columns ---")
for col in factor_cols:
    vals = df[col].dropna()
    if not pd.api.types.is_numeric_dtype(vals) or len(vals) == 0:
        continue
    if vals.min() >= -1 and vals.max() <= 1:
        print(f"  {col:<30s} range [{vals.min():.4f}, {vals.max():.4f}] — likely ratio (0-1)")
    elif vals.min() >= -100 and vals.max() <= 100:
        print(f"  {col:<30s} range [{vals.min():.2f}, {vals.max():.2f}] — possibly percentage")

# ── 3d. Check for constant or near-constant columns ─────────────────────────
print(f"\n--- Constant or near-constant columns ---")
for col in factor_cols:
    vals = df[col].dropna()
    if not pd.api.types.is_numeric_dtype(vals) or len(vals) == 0:
        continue
    if vals.nunique() <= 3:
        print(f"  {col:<30s} only {vals.nunique()} unique values: {sorted(vals.unique())}")
    elif abs(vals.mean()) > 1e-10 and vals.std() / abs(vals.mean()) < 0.01:
        print(f"  {col:<30s} near-constant (cv = {vals.std()/abs(vals.mean()):.6f})")

# ── 3e. Correlation between related columns ─────────────────────────────────
print(f"\n--- Net position check (net = long - short) ---")
trader_types = ['lev', 'am', 'dealer']
for t in trader_types:
    long_col = f'{t}_long'
    short_col = f'{t}_short'
    net_col = f'{t}_net'
    if all(c in df.columns for c in [long_col, short_col, net_col]):
        computed_net = df[long_col] - df[short_col]
        diff = (computed_net - df[net_col]).abs()
        valid = diff.dropna()
        if len(valid) > 0:
            max_diff = valid.max()
            if max_diff < 1:
                print(f"  ✓ {net_col} = {long_col} - {short_col} (max diff: {max_diff:.4f})")
            else:
                print(f"  ⚠ {net_col} ≠ {long_col} - {short_col} (max diff: {max_diff:,.0f})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS TO DROP:
   - Any column with ≥30% NaN
   - Constant or near-constant columns
   - Redundant columns (if net = long - short, keep net, drop components?)

2. DATE RANGE:
   - Trim to 2004+ if earlier data exists

3. available_date:
   - Keep for merge pipeline (publication lag awareness)
   - CFTC positions are as-of Tuesday, released Friday
   - When forward-filling to daily, use available_date not report date

4. NaN HANDLING:
   - Weekly macro data — forward-fill appropriate in merge pipeline
   - Same treatment as FRED weekly

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — CFTC

Shape: 969 rows × 24 columns
Date range: 2006-06-13 → 2024-12-31
Available date range: 2006-06-19 → 2025-01-06
Unique dates: 969

Factor columns (22):
    1. lev_long                       int64          
    2. lev_short                      int64          
    3. lev_spread                     int64          
    4. am_long                        int64          
    5. am_short                       int64          
    6. am_spread                      int64          
    7. dealer_long                    int64          
    8. dealer_short                   int64          
    9. dealer_spread                  int64          
   10. other_long                     int64          
   11. other_short                    int64          
   12. other_spread                   int64          
   13. open_interest                  int64          
   14. lev_net                        int64          
   15. am_net                         int64          
   16. 

In [2]:
display(df.columns)

Index(['date', 'lev_long', 'lev_short', 'lev_spread', 'am_long', 'am_short',
       'am_spread', 'dealer_long', 'dealer_short', 'dealer_spread',
       'other_long', 'other_short', 'other_spread', 'open_interest', 'lev_net',
       'am_net', 'dealer_net', 'lev_net_pct', 'am_net_pct', 'dealer_net_pct',
       'lev_am_ratio', 'lev_net_chg', 'am_net_chg', 'available_date'],
      dtype='object')

In [6]:
# %% [markdown]
# ## Stage 5: Clean & Save
#
# **Data overview:**
# Weekly CFTC Commitments of Traders positioning data for S&P 500 E-mini
# futures. 969 rows from 2006-06-13 to 2024-12-31. Pulled via the Socrata API
# during collection. Contains positioning by trader type (leveraged funds,
# asset managers, dealers, other) broken into long, short, spread, and net.
#
# **No columns dropped.** All 22 factor columns retained. The component
# columns (long, short, spread) and net columns are both kept — the model
# can select what's useful, and z-standardisation makes scale irrelevant.
# Net position consistency verified: `lev_net = lev_long - lev_short` with
# zero error for all three trader types.
#
# **`available_date` retained as metadata (not a factor):**
# The publication lag is exactly 6 days for every row — positions are as-of
# Tuesday, released the following Monday. The merge pipeline should use
# `available_date` (not `date`) when forward-filling to daily, to ensure
# point-in-time correctness: Tuesday's positions should only enter the model
# from the following Monday onwards.
#
# **Data starts June 2006, not 2004:**
# The CFTC Disaggregated report (which breaks out leveraged funds, asset
# managers, and dealers separately) only began in June 2006. Earlier CFTC
# data uses the legacy commercial/non-commercial breakdown which isn't
# comparable. This is the same situation as FRED `twexb`/`twexm` — structural
# late start, resolved when the merged dataset starts from 2006.
#
# **Structural NaN (2 values only):**
# `lev_net_chg` and `am_net_chg` are NaN on the first row (2006-06-13) —
# no prior week to compute a change from. Left as NaN.
#
# **Weekly structure is perfect:** 948 gaps of exactly 7 days, 10 gaps of
# 6 days and 10 of 8 days (holiday-shifted Tuesdays to Monday/Wednesday).
# No missing weeks, no duplicates. Forward-fill to daily in the merge pipeline.
#
# **No winsorisation applied.** All value ranges are sensible for S&P 500
# futures positioning. Leveraged funds are typically net short (928/969 weeks),
# asset managers typically net long (969/969 weeks). Handled in merge pipeline.
#
# **Factors retained: 22** (all kept)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 5: CLEAN & SAVE")
print("=" * 90)

# ── 5a. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]
if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN")
else:
    total_nan = nan_cols.sum()
    print(f"\n  Remaining NaN: {total_nan} (structural, left intentionally)")
    for col, n in nan_cols.items():
        print(f"    {col:<25s} {n} NaN")

# ── 5b. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Available date range: {df['available_date'].min().date()} → {df['available_date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    vals = df[c].dropna()
    nan_n = df[c].isna().sum()
    nan_str = f"  ({nan_n} NaN)" if nan_n > 0 else ""
    print(f"    {i:>3d}. {c:<25s} range: [{vals.min():,.2f}, {vals.max():,.2f}]{nan_str}")

print(f"\n  Sample (first 3 rows):")
print(df.head(3).to_string(index=False))

# ── 5c. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'cftc_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns (including available_date)")

print("\nCleaning complete.")

STAGE 5: CLEAN & SAVE

  Remaining NaN: 2 (structural, left intentionally)
    lev_net_chg               1 NaN
    am_net_chg                1 NaN

  Final shape: 969 rows × 24 columns
  Factor columns: 22
  Date range: 2006-06-13 → 2024-12-31
  Available date range: 2006-06-19 → 2025-01-06

  Factor list (22 columns):
      1. lev_long                  range: [81,745.00, 887,102.00]
      2. lev_short                 range: [223,618.00, 1,142,001.00]
      3. lev_spread                range: [11,244.00, 279,069.00]
      4. am_long                   range: [370,566.00, 1,778,863.00]
      5. am_short                  range: [60,243.00, 1,120,091.00]
      6. am_spread                 range: [35,152.00, 620,941.00]
      7. dealer_long               range: [76,692.00, 710,110.00]
      8. dealer_short              range: [149,569.00, 1,584,927.00]
      9. dealer_spread             range: [5,106.00, 668,165.00]
     10. other_long                range: [13,956.00, 391,694.00]
     11. 